# Exportação dos Artefatos Congelados do PreviTU V6 (HistGradientBoosting)

Este notebook (~25 segundos em CPU) reajusta os dois modelos vencedores **`HistGradientBoostingClassifier`** da tese (2011–2022) usando exatamente os hiperparâmetros e limiares consolidados em `resultados_tese_4paradigmas_2023_2024/summary_models_2023_2024.csv` e exporta para o Google Drive (`/Todos os arquivos GPT/previtu_v6_artefatos/`):

- `previtu_v6_hgb_sem_gram.joblib` + `model_card_v6_sem_gram.json` (com SHA-256)
- `previtu_v6_hgb_com_gram.joblib` + `model_card_v6_com_gram.json` (com SHA-256)
- `predicoes_2023_2024_v6_*.npy` e `amostra_auditoria_2023_2024_v6_*.csv.gz` (para rodar `teste_previtu_core_v6.py`)

Depois, basta copiar esses arquivos + `previtu_core_v6.py` + `teste_previtu_core_v6.py` para a pasta `PreviTU_V5/predictor/v5/` do seu site.


In [ ]:
!pip install -q scikit-learn==1.7.2

In [ ]:
import hashlib
import json
import re
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from google.colab import drive
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

drive.mount("/content/drive", force_remount=False)

INPUT_PATH = "/content/drive/MyDrive/Todos os arquivos GPT/UROT_UROC_GG_Base_Definitiva_2011_2024_2026-10-19.csv.gz"
OUT_DIR = Path("/content/drive/MyDrive/Todos os arquivos GPT/previtu_v6_artefatos")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MAPA_CRUZES = {
    "NEGATIVO": 0.0, "NEGATIVA": 0.0, "AUSENTE": 0.0, "NORMAL": 0.0, "NAO REAGENTE": 0.0,
    "TRACOS": 0.5, "TRAÇOS": 0.5, "TRACO": 0.5, "TRAÇO": 0.5, "VESTIGIOS": 0.5, "VESTÍGIOS": 0.5,
    "ESCASSA": 1.0, "ESCASSO": 1.0, "1+": 1.0, "+": 1.0, "LEVE": 1.0, "DISCRETA": 1.0, "DISCRETO": 1.0, "POSITIVO": 1.0,
    "2+": 2.0, "++": 2.0, "MODERADA": 2.0, "MODERADO": 2.0,
    "3+": 3.0, "+++": 3.0, "ACENTUADA": 3.0, "ACENTUADO": 3.0, "NUMEROSA": 3.0, "INTENSA": 3.0,
    "4+": 4.0, "++++": 4.0, "MACICA": 4.0, "MACIÇA": 4.0,
}

def parse_densidade(val):
    if pd.isna(val): return np.nan
    s = str(val).strip().upper().replace(",", ".")
    m = re.search(r"(\d+(?:\.\d+)?)", s)
    if not m: return np.nan
    num = float(m.group(1))
    if 1000.0 <= num <= 1060.0: num /= 1000.0
    return num if 1.000 <= num <= 1.060 else np.nan

def parse_sedimento(val):
    if pd.isna(val): return np.nan
    s = str(val).strip().upper()
    if any(w in s for w in ["AUSENT", "NEGAT", "RARO"]): return 0.0
    if "NUMEROS" in s or "MACIC" in s or "INCONT" in s: return 100.0
    m = re.search(r"(\d+(?:[.,]\d+)?)", s)
    return float(m.group(1).replace(",", ".")) if m else np.nan

def parse_semiquantitativo(serie):
    s = serie.astype(str).str.strip().str.upper()
    num = pd.to_numeric(s.str.replace(",", ".", regex=False), errors="coerce")
    return num.fillna(s.map(MAPA_CRUZES))

def extrair_features_gram(serie_gram):
    out = pd.DataFrame(index=serie_gram.index)
    s = serie_gram.astype(str).str.upper().str.strip()
    invalido = serie_gram.isna() | s.isin(["", "NAN", "NONE", "NAO REALIZADO", "NÃO REALIZADO"])
    negativo = s.str.contains(r"NAO FORAM OBSERVAD|NÃO FORAM OBSERVAD|AUSENCIA DE BACT|AUSÊNCIA DE BACT|NEGATIV", regex=True)
    out["gram_bacterias_presentes"] = np.where(invalido, np.nan, np.where(negativo, 0.0, 1.0))
    out["gram_bgn"] = np.where(invalido, np.nan, s.str.contains(r"GRAM.?NEGATIV|BGN|BACILO", regex=True).astype(float))
    out["gram_cgp"] = np.where(invalido, np.nan, s.str.contains(r"GRAM.?POSITIV|CGP|COCO", regex=True).astype(float))
    out["gram_levedura"] = np.where(invalido, np.nan, s.str.contains(r"LEVEDUR|FUNG|PSEUDO.?HIFA|CANDIDA", regex=True).astype(float))
    return out

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

df = pd.read_csv(INPUT_PATH, sep=";", low_memory=False)
df.columns = [c.strip() for c in df.columns]
df["Cultura_positiva"] = pd.to_numeric(df["Cultura_positiva"], errors="coerce")
df["Ano"] = pd.to_numeric(df["Ano"], errors="coerce")
df = df.dropna(subset=["Cultura_positiva", "Paciente_ID", "Ano"]).copy()
df["Cultura_positiva"] = df["Cultura_positiva"].astype(int)
df["Ano"] = df["Ano"].astype(int)
df["Paciente_ID"] = df["Paciente_ID"].astype(str).str.strip()

df["UROT__GRAVID"] = df["UROT__GRAVID"].apply(parse_densidade)
df["UROT__GRAVID_DESVIO_QUAD"] = (df["UROT__GRAVID"] - 1.015) ** 2
for c in ["Idade_anos", "UROT__PH"]:
    df[c] = pd.to_numeric(df[c].astype(str).str.replace(",", ".", regex=False), errors="coerce")
for c in ["UROT__SEDLEU", "UROT__SEDHEM"]:
    df[c] = df[c].apply(parse_sedimento)

hemogl_col = next(c for c in ["UROT__HEMOGL", "UROT__HEMOGLOBINA", "UROT__HEMOG", "UROT__HB", "UROT__SANGUE"] if c in df.columns)
for c in ["UROT__ESTERASE", "UROT__GLIC", "UROT__FLORA", hemogl_col]:
    df[c] = parse_semiquantitativo(df[c])

CAT_COLS = ["Sexo", "Origem", "UROT__NITRIT"]
for c in CAT_COLS:
    s = df[c].astype(str).str.strip()
    df[c] = np.where(df[c].isna() | s.isin(["", "nan", "NaN", "None", "NONE", "<NA>"]), np.nan, s)

gram_df = extrair_features_gram(df["Gram_texto"])
for gcol in ["gram_bacterias_presentes", "gram_bgn", "gram_cgp", "gram_levedura"]:
    df[gcol] = gram_df[gcol]

dev_df = df[df["Ano"] <= 2022].copy().reset_index(drop=True)
seen = set(dev_df["Paciente_ID"].unique())
test_df = df[df["Ano"].isin([2023, 2024]) & (~df["Paciente_ID"].isin(seen))].copy().reset_index(drop=True)

NUM_BASE = [
    "Idade_anos", "UROT__GRAVID", "UROT__GRAVID_DESVIO_QUAD", "UROT__PH",
    "UROT__GLIC", "UROT__SEDHEM", "UROT__SEDLEU", "UROT__ESTERASE", "UROT__FLORA", hemogl_col,
]
GRAM_COLS = ["gram_bacterias_presentes", "gram_bgn", "gram_cgp", "gram_levedura"]

# Hiperparâmetros e limiares congelados do experimento oficial (summary_models_2023_2024.csv)
SPECS = {
    "sem_gram": {
        "num_cols": NUM_BASE,
        "params": {"learning_rate": 0.04, "max_leaf_nodes": 31, "min_samples_leaf": 60, "max_iter": 250, "random_state": 42},
        "limiares": {"youden": 0.20343186387277354, "sensibilidade_95": 0.05357552116357956},
    },
    "com_gram": {
        "num_cols": NUM_BASE + GRAM_COLS,
        "params": {"learning_rate": 0.08, "max_leaf_nodes": 31, "min_samples_leaf": 60, "max_iter": 250, "random_state": 42},
        "limiares": {"youden": 0.21141896319199643, "sensibilidade_95": 0.04846699003955088},
    },
}

for via, sp in SPECS.items():
    num_cols = sp["num_cols"]
    feats = num_cols + CAT_COLS

    prep = ColumnTransformer([
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median", add_indicator=True))]), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="AUSENTE")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), CAT_COLS),
    ], remainder="drop")

    pipe = Pipeline([("prep", prep), ("model", HistGradientBoostingClassifier(**sp["params"]))])
    pipe.fit(dev_df[feats], dev_df["Cultura_positiva"].to_numpy())
    probs = pipe.predict_proba(test_df[feats])[:, 1]
    y_te = test_df["Cultura_positiva"].to_numpy()

    auroc = float(roc_auc_score(y_te, probs))
    prauc = float(average_precision_score(y_te, probs))
    brier = float(brier_score_loss(y_te, probs))

    art_path = OUT_DIR / f"previtu_v6_hgb_{via}.joblib"
    joblib.dump({
        "versao": f"V6_HGB_{via.upper()}_2026-11-24",
        "pipeline": pipe,
        "variaveis_numericas": num_cols,
        "variaveis_categoricas": CAT_COLS,
        "limiares": sp["limiares"],
        "hiperparametros": sp["params"],
        "sklearn_version": sklearn.__version__,
    }, art_path, compress=3)

    sha = sha256_file(art_path)
    np.save(OUT_DIR / f"predicoes_2023_2024_v6_{via}.npy", probs)
    test_df[feats + ["Cultura_positiva"]].to_csv(
        OUT_DIR / f"amostra_auditoria_2023_2024_v6_{via}.csv.gz", index=False, compression="gzip"
    )

    card = {
        "versao": f"V6_HGB_{via.upper()}_2026-11-24",
        "via": via,
        "algoritmo": "HistGradientBoostingClassifier",
        "gerado_em_utc": datetime.now(timezone.utc).isoformat(),
        "sklearn_version": sklearn.__version__,
        "coorte_desenvolvimento": {"periodo": "2011-2022", "n_pedidos": int(len(dev_df))},
        "coorte_teste_temporal": {"periodo": "2023-2024 (pacientes novos)", "n_pedidos": int(len(test_df))},
        "variaveis_numericas": num_cols,
        "variaveis_categoricas": CAT_COLS,
        "hiperparametros": sp["params"],
        "limiares": sp["limiares"],
        "desempenho_2023_2024": {"auroc": auroc, "pr_auc": prauc, "brier": brier},
        "hashes": {"modelo_sha256": sha},
    }
    (OUT_DIR / f"model_card_v6_{via}.json").write_text(json.dumps(card, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"[{via}] Exportado! AUROC={auroc:.4f} | PR-AUC={prauc:.4f} | Brier={brier:.4f} | SHA256={sha[:12]}...")

print("Todos os artefatos salvos em:", OUT_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[sem_gram] Exportado! AUROC=0.8863 | PR-AUC=0.8000 | Brier=0.0921 | SHA256=462aded6ff5d...
[com_gram] Exportado! AUROC=0.9132 | PR-AUC=0.8621 | Brier=0.0693 | SHA256=94e97bb9e72e...
Todos os artefatos salvos em: /content/drive/MyDrive/Todos os arquivos GPT/previtu_v6_artefatos


# LINHA DE CÓDIGO QUE CHAMA A AUDITORIA

In [ ]:
!python "/content/drive/MyDrive/Todos os arquivos GPT/previtu_v6_artefatos/teste_previtu_core_v6_exact.py"


=== AUDITORIA VIA: sem_gram (EAS fisiológico (9 parâmetros) + demografia/origem (sem Gram)) ===
[1] Lote 2023–2024 (n=9425): diferença máxima = 0.000e+00
[2] 200 registros pela interface pública: diferença máxima = 0.000e+00
[3] UROT__SEDLEU ausente=0.184784 vs. zero=0.070443
[4] Limiar=0.053576 | Versão=V6_HGB_SEM_GRAM_2026-11-24 | SHA256=462aded6ff5d...
[5] AUROC 2023–2024 recalculada = 0.886290 (card = 0.886290)

=== AUDITORIA VIA: com_gram (EAS fisiológico (9 parâmetros) + demografia/origem + Gram morfológico (4 marcadores)) ===
[1] Lote 2023–2024 (n=9425): diferença máxima = 0.000e+00
[2] 200 registros pela interface pública: diferença máxima = 0.000e+00
[3] UROT__SEDLEU ausente=0.872219 vs. zero=0.750911
[4] Limiar=0.048467 | Versão=V6_HGB_COM_GRAM_2026-11-24 | SHA256=94e97bb9e72e...
[5] AUROC 2023–2024 recalculada = 0.913176 (card = 0.913176)

RESULTADO FINAL: TODOS OS TESTES PASSARAM


# CORREÇÃO DAS FALHAS ENCONTRADAS E NOVA AUDITORIA

In [ ]:
from pathlib import Path

p = Path("/content/drive/MyDrive/Todos os arquivos GPT/previtu_v6_artefatos/previtu_core_v6.py")
txt = p.read_text(encoding="utf-8")

txt = txt.replace(
    'num = pd.to_numeric(str(v).replace(",", "."), errors="coerce")',
    'num = float(v) if isinstance(v, (int, float, np.number)) else pd.to_numeric(str(v).replace(",", "."), errors="coerce")',
)
txt = txt.replace(
    'g = pd.to_numeric(str(dados["UROT__GRAVID"]).replace(",", "."), errors="coerce")',
    'g = float(dados["UROT__GRAVID"]) if isinstance(dados["UROT__GRAVID"], (int, float, np.number)) else pd.to_numeric(str(dados["UROT__GRAVID"]).replace(",", "."), errors="coerce")',
)
p.write_text(txt, encoding="utf-8")

!python "/content/drive/MyDrive/Todos os arquivos GPT/previtu_v6_artefatos/teste_previtu_core_v6_exact.py"


=== AUDITORIA VIA: sem_gram (EAS fisiológico (9 parâmetros) + demografia/origem (sem Gram)) ===
[1] Lote 2023–2024 (n=9425): diferença máxima = 0.000e+00
[2] 200 registros pela interface pública: diferença máxima = 0.000e+00
[3] UROT__SEDLEU ausente=0.184784 vs. zero=0.070443
[4] Limiar=0.053576 | Versão=V6_HGB_SEM_GRAM_2026-11-24 | SHA256=b22e21d0a5d8...
[5] AUROC 2023–2024 recalculada = 0.886290 (card = 0.886290)

=== AUDITORIA VIA: com_gram (EAS fisiológico (9 parâmetros) + demografia/origem + Gram morfológico (4 marcadores)) ===
[1] Lote 2023–2024 (n=9425): diferença máxima = 0.000e+00
[2] 200 registros pela interface pública: diferença máxima = 0.000e+00
[3] UROT__SEDLEU ausente=0.872219 vs. zero=0.750911
[4] Limiar=0.048467 | Versão=V6_HGB_COM_GRAM_2026-11-24 | SHA256=9339ec28c6f5...
[5] AUROC 2023–2024 recalculada = 0.913176 (card = 0.913176)

RESULTADO FINAL: TODOS OS TESTES PASSARAM
